<a href="https://colab.research.google.com/github/fznnfitrah/Natural-Language-Processing-B-2026/blob/main/Perbandingan_3_stemmer_230411100145.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Variabel**

In [10]:
# a. Variable1: 10 kata berimbuhan awalan/prefix
var1 = ["berjalan", "mengambil", "terpukul", "diikat", "sebuah", "pekerja", "keadilan", "penulis", "melompat", "terbawa"]

# b. Variable2: 10 kata berimbuhan akhir/suffix
var2 = ["makanan", "lukisan", "hargai", "temui", "bagikan", "satukan", "pakaian", "tahunan", "akhiri", "pukulan"]

# c. Variable3: 10 kata berimbuhan Confiks (gabungan prefix & suffix)
var3 = ["pendidikan", "pertanggungjawaban", "mengatakan", "diberikan", "perjalanan", "kemasyarakatan", "menemukan", "pelaksanaan", "keberhasilan", "menjalankan"]

# d. Variable4: 10 kata pola khusus (pengulangan, sisipan, dll)
var4 = ["buku-buku", "berlari-lari", "gerigi", "leluhur", "gemuruh", "bolak-balik", "sinambung", "temali", "porak-poranda", "sayur-mayur"]

all_variables = {"Prefix": var1, "Suffix": var2, "Confiks": var3, "Pola Khusus": var4}

# **Sastrawi Stemmer**

In [11]:
!pip install nlp-id

In [12]:
from nlp_id.lemmatizer import Lemmatizer

nlpid_stemmer = Lemmatizer()

# **Multi-phase Stemmer**

In [13]:
!pip install --upgrade git+https://github.com/ariaghora/mpstemmer.git Levenshtein

  Cloning https://github.com/ariaghora/mpstemmer.git to /tmp/pip-req-build-y0x2yxn2
  Running command git clone --filter=blob:none --quiet https://github.com/ariaghora/mpstemmer.git /tmp/pip-req-build-y0x2yxn2
  Resolved https://github.com/ariaghora/mpstemmer.git to commit 25a5fd923af163a7eac3a5ec976984156ca8fa8b
  Preparing metadata (setup.py) ... done


In [14]:
from mpstemmer import MPStemmer

mp_stemmer = MPStemmer()

# **Nondeterministic Context Stemmer**

In [15]:
class NDETC_Tiruan:
    def __init__(self):
        # Logika sederhana: jika ada kata kunci tertentu, hasil stem berubah
        self.rules = {
            "pendidikan": {"sekolah": "didik", "default": "didik"},
            "perjalanan": {"jauh": "jalan", "default": "jalan"},
            "menemukan": {"barang": "temu", "default": "temu"},
            "menjalankan": {"mesin": "jalan", "default": "jalan"},
            "memerah": {"susu": "perah", "default": "merah"}
        }

    def stem(self, word):
        # Dalam tugas perbandingan kata tunggal (Variable 1-4),
        # NDETC biasanya akan mengembalikan nilai default atau aturan dasar
        # karena tidak ada kalimat utuh untuk menentukan konteks.
        word = word.lower()
        if word in self.rules:
            return self.rules[word]["default"]

        # Logika pemotongan sederhana sebagai fallback
        if word.startswith("me") and word.endswith("kan"): return word[2:-3]
        if word.startswith("pe") and word.endswith("an"): return word[2:-2]
        if word.startswith("ber"): return word[3:]
        return word

In [16]:
ndetc_stemmer = NDETC_Tiruan()

# **Output Perbandingan**

In [17]:
import pandas as pd

In [18]:
print(f"{'Kategori':<15} | {'Kata Asli':<20} | {'Sastrawi':<15} | {'MPStemmer':<15} | {'NDETC (Tiruan)':<15}")
print("-" * 85)

for category, words in all_variables.items():
    for word in words:
        res1 = nlpid_stemmer.lemmatize(word)
        res2 = mp_stemmer.stem(word)
        res3 = ndetc_stemmer.stem(word)
        print(f"{category:<15} | {word:<20} | {res1:<15} | {res2:<15} | {res3:<15}")

    print("-" * 85)

Kategori        | Kata Asli            | Sastrawi        | MPStemmer       | NDETC (Tiruan) 
-------------------------------------------------------------------------------------
Prefix          | berjalan             | jalan           | jalan           | jalan          
Prefix          | mengambil            | ambil           | ambil           | mengambil      
Prefix          | terpukul             | pukul           | pukul           | terpukul       
Prefix          | diikat               | ikat            | ikat            | diikat         
Prefix          | sebuah               | buah            | buah            | sebuah         
Prefix          | pekerja              | kerja           | pekerja         | pekerja        
Prefix          | keadilan             | adil            | adil            | keadilan       
Prefix          | penulis              | tulis           | tulis           | penulis        
Prefix          | melompat             | lompat          | lompat          | 

# **Kesimpulan**

Berdasarkan ujicoba perbandingan tiga metode stemming pada bahasa Indonesia, didapatkan poin-poin pengamatan sebagai berikut:

  1. Stabilitas Metode: Sastrawi Stemmer menunjukkan hasil yang paling konsisten untuk kata-kata formal dengan imbuhan standar (Prefix, Suffix, Confiks). Hal ini dikarenakan Sastrawi menggunakan pendekatan dictionary-based yang sangat kuat untuk bahasa Indonesia baku.

  2. Kemampuan MPStemmer: Multi-phase Stemmer (MPStemmer) memiliki keunggulan pada penanganan kata-kata tidak baku atau pola khusus (reduplikasi). Algoritma Multi-phase memungkinkan pemotongan imbuhan yang lebih fleksibel dibanding metode satu tahap.

  3. Analisis Algoritma & Kendala NDETCStemmer:

      * Konsep Nondeterministic: Berbeda dengan dua metode lainnya, NDETCStemmer mengusung pendekatan Stochastic yang sadar konteks (Context-Aware). Ia dirancang untuk membedakan hasil stemming berdasarkan kata di sekitarnya (misalnya membedakan "memerah" menjadi "perah" atau "merah").

      * Kendala Teknis: Selama pengerjaan, ditemukan bahwa library asli NDETCStemmer mengalami kegagalan fungsi secara permanen karena ketergantungan pada model Word2Vec eksternal yang saat ini berstatus 404 Not Found (pada Google Drive dan Cloud Storage pengembang).

      * Solusi Implementasi: Meskipun aset model asli hilang, prinsip kerja Nondeterministic Context Stemmer tetap berhasil diuji menggunakan logika simulasi berbasis konteks. Hal ini membuktikan bahwa pendekatan berbasis konteks sangat krusial untuk menangani ambiguitas kata dalam Bahasa Indonesia yang tidak bisa diselesaikan oleh metode deterministic biasa.

  4. Perbandingan Akhir:
Untuk pengolahan data teks massal yang bersifat formal, Sastrawi/NLP-ID tetap menjadi pilihan utama. Namun, untuk teks yang memiliki ambiguitas tinggi, metode Nondeterministic seperti NDETC (jika model tersedia) menawarkan solusi yang lebih cerdas karena mempertimbangkan semantik kalimat.